# Milestone 1 - Préparation et nettoyage des données

**But :** transformer les trois sources brutes (`released_schedule.csv`, `distances.csv`, `rate_sheet.csv`)
en entrées propres et fiables pour la consolidation. Chaque cas trouvé dans `data_quality.ipynb`
reçoit une règle de nettoyage explicite, un identifiant et une ligne dans le **registre des anomalies** (EF02).
Chaque imputation est tracée dans une colonne dédiée (EF03).

Colonnes ignorées : `transmission_date`, `version_date` (consigne du prof).

| ID | Source | Cas | Détection | Règle de nettoyage | Statut |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **DQ01** | planning | Identifiants mal formés / espaces parasites | Expression régulière par colonne d'identifiant | Retirer les espaces et passer en majuscules ; supprimer les lignes qui ne correspondent toujours pas (impossibles à rattacher) | Appliqué |
| **DQ02** | planning | Doublons | Doublons exacts, puis doublons sur la clé métier | Supprimer les doublons exacts ; signaler et garder ceux dont la clé est identique mais les valeurs différentes | Appliqué |
| **DQ03** | planning | Quantité = 0, négative ou vide | `schedule_quantity <= 0` ou NaN | Supprimer la ligne : 0 signifie « pas de besoin », négatif / vide n'ont pas de sens | Appliqué |
| **DQ04** | planning | Date hors horizon, en week-end ou un jour férié allemand | Comparaison avec l'horizon, `day_of_week`, calendrier des jours fériés (`holidays`) | Signaler et conserver : on ne reçoit rien le week-end ni les jours fériés, mais la ligne reste et l'optimisation la prend en compte | Appliqué |
| **DQ05** | planning | Dates >= 01/01/2027 = besoins cumulés après l'horizon journalier (jour férié) | `date >= tail_date` | Supprimer ces lignes (date de coupure réglable via `tail_date`) | Appliqué |
| **DQ06** | planning | Poids / volume manquant ou à 0 avec quantité > 0 | `isna()` ou `<= 0` | Médiane du même couple fournisseur × usine, sinon médiane de la même usine ; marquer `*_imputed` et `*_impute_level` | Appliqué |
| **DQ07** | planning | Ligne dont le poids dépasse la capacité d'un camion (24,5 t) | `weight > truck_kg` | Supprimer la ligne (capacité réglable via `truck_kg`) | Appliqué |
| **DQ08** | distances | Fournisseur en double | `duplicated("supplier_id")` | Garder la première valeur, signaler le reste | Appliqué |
| **DQ09** | distances | Fournisseur sans distance, absent du fichier ou distance <= 0 | `isna()`, jointure gauche, `<= 0` | Médiane des distances des fournisseurs présents dans le planning ; marquer `distance_imputed` | Appliqué |
| **DQ10** | grille tarifaire | Prix manquant, prix qui baisse quand la distance ou le poids augmente | `isna()`, `diff() < 0` | Arrêter (erreur) : la grille doit être complète et monotone | Appliqué |

Les paramètres sont regroupés à un seul endroit (`CleaningConfig`, EF12) pour pouvoir les changer sans toucher à la logique.

In [1]:
from __future__ import annotations

import logging
from dataclasses import dataclass
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-7s %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)
logger = logging.getLogger("preparation")


@dataclass(frozen=True)
class CleaningConfig:
    """All tunable parameters of the preparation step (EF12)."""

    data_dir: Path = Path("../docs")
    output_dir: Path = Path("../data")
    horizon_start: str = "2026-11-09"
    horizon_end: str = "2027-01-01"
    truck_kg: float = 24_500.0  # DQ07: a line heavier than one truck is removed
    holiday_country: str = "DE"  # no reception on German public holidays
    holiday_subdiv: str | None = None  # e.g. "BY" to add regional holidays
    tail_date: str = "2027-01-01"  # DQ05: rows dated on or after this day are removed
    key_columns: tuple[str, ...] = (
        "date",
        "supplier_id",
        "material",
        "plant",
        "unloading_point",
    )


cfg = CleaningConfig()
logger.info("Configuration: %s", cfg)

11:54:13 INFO    Configuration: CleaningConfig(data_dir=WindowsPath('../docs'), output_dir=WindowsPath('../data'), horizon_start='2026-11-09', horizon_end='2027-01-01', truck_kg=24500.0, holiday_country='DE', holiday_subdiv=None, tail_date='2027-01-01', key_columns=('date', 'supplier_id', 'material', 'plant', 'unloading_point'))


## 0. Registre des anomalies (EF02)

Chaque règle ci-dessous appelle `register(...)` : source, identifiant de la règle, gravité, nombre de lignes et action prise.
Le registre est exporté à la fin.

In [2]:
ANOMALIES: list[dict[str, object]] = []


def register(rule: str, source: str, severity: str, n_rows: int, action: str) -> None:
    """Record one finding in the register and log it with the matching level."""
    ANOMALIES.append(
        {
            "rule": rule,
            "source": source,
            "severity": severity if n_rows else "none",
            "rows": int(n_rows),
            "action": action,
        }
    )
    level = logging.WARNING if n_rows and severity != "info" else logging.INFO
    logger.log(level, "%s [%s] %d row(s) - %s", rule, source, n_rows, action)

## 1. Chargement des sources brutes

In [3]:
schedule_raw = pd.read_csv(
    cfg.data_dir / "released_schedule.csv", encoding="utf-8-sig", parse_dates=["date"]
).drop(columns=["transmission_date", "version_date"])
distances_raw = pd.read_csv(cfg.data_dir / "distances.csv", encoding="utf-8-sig")
rates_raw = pd.read_csv(cfg.data_dir / "rate_sheet.csv", dtype={"km_up_to": str})

logger.info("released_schedule: %s", schedule_raw.shape)
logger.info("distances: %s", distances_raw.shape)
logger.info("rate_sheet: %s", rates_raw.shape)

schedule = schedule_raw.copy()
schedule_raw.head()

11:54:13 INFO    released_schedule: (5728, 8)
11:54:13 INFO    distances: (156, 2)
11:54:13 INFO    rate_sheet: (14, 16)


,date,supplier_id,material,plant,unloading_point,schedule_quantity,weight,volume
0,2026-11-10,SUP_00091,MAT_000110,PLT_004,ULP_0050,0.0,0.0,0.4116
1,2026-11-11,SUP_00091,MAT_000110,PLT_004,ULP_0050,5000.0,92.6,0.4116
2,2026-11-12,SUP_00091,MAT_000110,PLT_004,ULP_0050,0.0,0.0,0.4116
3,2026-11-13,SUP_00091,MAT_000110,PLT_004,ULP_0050,5000.0,92.6,0.4116
4,2026-11-16,SUP_00091,MAT_000110,PLT_004,ULP_0050,5000.0,92.6,0.4116


## 2. Identifiants

### DQ01 - Identifiants mal formés
**Règle :** retirer les espaces et passer chaque identifiant en majuscules, puis supprimer les lignes qui ne respectent toujours pas le format attendu.
**Pourquoi :** une ligne avec un fournisseur ou une usine inconnu ne peut pas être rattachée à un couple fournisseur × usine (RG01).

In [4]:
ID_PATTERNS: dict[str, str] = {
    "supplier_id": r"SUP_\d{5}",
    "material": r"MAT_\d{6}",
    "plant": r"PLT_\d{3}",
    "unloading_point": r"ULP_\d{4}",
}

n_reformatted = 0
invalid = pd.Series(False, index=schedule.index)
for column, pattern in ID_PATTERNS.items():
    cleaned = schedule[column].astype("string").str.strip().str.upper()
    n_reformatted += int((cleaned != schedule[column]).sum())
    schedule[column] = cleaned
    invalid |= ~cleaned.str.fullmatch(pattern).fillna(False)

register("DQ01", "schedule", "warning", n_reformatted, "trimmed / upper-cased")
register("DQ01", "schedule", "error", int(invalid.sum()), "dropped (invalid identifier)")
schedule = schedule.loc[~invalid].copy()

schedule.head()

11:54:13 INFO    DQ01 [schedule] 0 row(s) - trimmed / upper-cased
11:54:13 INFO    DQ01 [schedule] 0 row(s) - dropped (invalid identifier)


,date,supplier_id,material,plant,unloading_point,schedule_quantity,weight,volume
0,2026-11-10,SUP_00091,MAT_000110,PLT_004,ULP_0050,0.0,0.0,0.4116
1,2026-11-11,SUP_00091,MAT_000110,PLT_004,ULP_0050,5000.0,92.6,0.4116
2,2026-11-12,SUP_00091,MAT_000110,PLT_004,ULP_0050,0.0,0.0,0.4116
3,2026-11-13,SUP_00091,MAT_000110,PLT_004,ULP_0050,5000.0,92.6,0.4116
4,2026-11-16,SUP_00091,MAT_000110,PLT_004,ULP_0050,5000.0,92.6,0.4116


## 3. Doublons

### DQ02
**Règle :** supprimer les doublons exacts. Les lignes qui partagent la clé métier (`date, fournisseur, material, usine, quai`)
mais ont des valeurs différentes sont signalées et conservées : les additionner masquerait une erreur de la source.
**Pourquoi :** `data_quality.ipynb` n'en a trouvé aucun, mais la règle protège la prochaine livraison de fichier.

In [5]:
n_exact = int(schedule.duplicated().sum())
schedule = schedule.drop_duplicates()
register("DQ02", "schedule", "warning", n_exact, "dropped (exact duplicate)")

n_key = int(schedule.duplicated(list(cfg.key_columns), keep=False).sum())
register("DQ02", "schedule", "error", n_key, "flagged only (same key, different values)")

11:54:13 INFO    DQ02 [schedule] 0 row(s) - dropped (exact duplicate)
11:54:13 INFO    DQ02 [schedule] 0 row(s) - flagged only (same key, different values)


## 4. Quantités et calendrier

### DQ03 - Quantité = 0, négative ou vide
**Règle :** ne garder que les lignes avec `schedule_quantity > 0`.
**Pourquoi :** 0 signifie « pas de besoin » (dictionnaire du cours), il n'y a donc rien à transporter. Une quantité négative ou vide
n'a pas de sens métier. Ces lignes sont comptées par type avant d'être supprimées.

### DQ04 - Date hors horizon, en week-end ou un jour férié allemand
**Règle :** signaler et **conserver** la ligne.
**Pourquoi :** on ne reçoit rien le week-end ni les jours fériés allemands, mais un besoin daté ces jours-là reste un vrai besoin :
l'optimisation le prend en compte (il sera livré un jour ouvré précédent, RG10). Une date hors de l'horizon 09/11/2026 - 01/01/2027 est seulement signalée.
Les jours fériés viennent du paquet `holidays` (Allemagne, niveau national par défaut ; une région se règle avec `holiday_subdiv`).
Sur l'horizon, ce sont le 25/12 et le 26/12 (et le 01/01, retiré par DQ05).

In [6]:
qty = schedule["schedule_quantity"]
register("DQ03", "schedule", "info", int((qty == 0).sum()), "dropped (quantity = 0, no need)")
register("DQ03", "schedule", "error", int((qty < 0).sum()), "dropped (negative quantity)")
register("DQ03", "schedule", "error", int(qty.isna().sum()), "dropped (missing quantity)")
schedule = schedule.loc[qty > 0].copy()

outside = ~schedule["date"].between(cfg.horizon_start, cfg.horizon_end)
weekend = schedule["date"].dt.day_of_week >= 5
register("DQ04", "schedule", "warning", int(outside.sum()), "flagged and kept (outside horizon)")
register("DQ04", "schedule", "warning", int(weekend.sum()), "flagged and kept (no delivery on weekends, handled by the optimization)")

german_holidays = holidays.country_holidays(
    cfg.holiday_country,
    subdiv=cfg.holiday_subdiv,
    years=range(schedule["date"].dt.year.min(), schedule["date"].dt.year.max() + 1),
)
# Rows on tail_date and after are removed by DQ05: do not report them here
on_holiday = schedule["date"].dt.date.isin(list(german_holidays)) & (
    schedule["date"] < pd.Timestamp(cfg.tail_date)
)
register(
    "DQ04",
    "schedule",
    "warning",
    int(on_holiday.sum()),
    "flagged and kept (no delivery on German public holidays, handled by the optimization)",
)
logger.info("Rows with a need: %d", len(schedule))

11:54:13 INFO    DQ03 [schedule] 959 row(s) - dropped (quantity = 0, no need)
11:54:13 INFO    DQ03 [schedule] 0 row(s) - dropped (negative quantity)
11:54:13 INFO    DQ03 [schedule] 0 row(s) - dropped (missing quantity)
11:54:13 INFO    DQ04 [schedule] 0 row(s) - flagged and kept (outside horizon)
11:54:13 INFO    DQ04 [schedule] 0 row(s) - flagged and kept (no delivery on weekends, handled by the optimization)
11:54:14 INFO    DQ04 [schedule] 0 row(s) - flagged and kept (no delivery on German public holidays, handled by the optimization)
11:54:14 INFO    Rows with a need: 4769


### DQ05 - Dates >= 01/01/2027
**Règle :** supprimer toutes les lignes dont la date est supérieure ou égale à `tail_date` (01/01/2027 par défaut).
La date de coupure se règle dans `CleaningConfig` ; le nombre de lignes retirées est tracé dans le registre.
**Pourquoi :** c'est un jour férié et il regroupe les besoins *après* l'horizon journalier (quantités plusieurs fois supérieures
à un jour normal, 55 materials jamais vus avant). Le garder fausserait le remplissage et le lissage.
**Alternative :** repousser `tail_date` (par exemple `2027-01-02`) pour le garder comme un jour normal, cohérent avec les chiffres du cours (1 731 livraisons).

In [7]:
is_tail = schedule["date"] >= pd.Timestamp(cfg.tail_date)
register(
    "DQ05",
    "schedule",
    "info",
    int(is_tail.sum()),
    f"dropped (date >= {cfg.tail_date}, needs accumulated after the horizon)",
)
schedule = schedule.loc[~is_tail].copy()
logger.info("Rows kept after DQ05: %d", len(schedule))

11:54:14 INFO    DQ05 [schedule] 379 row(s) - dropped (date >= 2027-01-01, needs accumulated after the horizon)
11:54:14 INFO    Rows kept after DQ05: 4390


## 5. Poids et volume

### DQ06 - Poids / volume manquant ou à 0
**Règle**, appliquée dans cet ordre, avec `quantity > 0` :
1. médiane du **même couple fournisseur × usine** (l'unité de consolidation) ;
2. sinon médiane de la **même usine** ;
3. sinon la ligne est signalée en erreur (ne devrait pas arriver).

Les médianes n'utilisent que des valeurs valides (`> 0`) de l'horizon journalier (le 01/01 est déjà retiré par DQ05).
Chaque valeur imputée est tracée dans `weight_imputed` / `volume_imputed` et `*_impute_level` (EF03).

**Pourquoi :** supprimer la ligne ferait oublier un vrai besoin. Une règle de trois sur la quantité n'est pas fiable, car le
poids est une partie fixe (emballage) plus une partie par pièce, il n'est donc pas proportionnel.

In [8]:
def impute_with_median(df: pd.DataFrame, column: str) -> pd.DataFrame:
    """Fill missing or non-positive values of `column` with a couple, then plant median."""
    result = df.copy()
    needs = result[column].isna() | (result[column] <= 0)

    reference = result[column].where(result[column] > 0)
    couple_median = reference.groupby([result["supplier_id"], result["plant"]]).transform("median")
    plant_median = reference.groupby(result["plant"]).transform("median")

    by_couple = needs & couple_median.notna()
    by_plant = needs & ~by_couple & plant_median.notna()
    unresolved = needs & ~by_couple & ~by_plant

    level = pd.Series(pd.NA, index=result.index, dtype="string")
    level[by_couple] = "couple"
    level[by_plant] = "plant"

    result.loc[by_couple, column] = couple_median[by_couple]
    result.loc[by_plant, column] = plant_median[by_plant]
    result[f"{column}_imputed"] = needs & ~unresolved
    result[f"{column}_impute_level"] = level

    register(
        "DQ06",
        "schedule",
        "warning",
        int(by_couple.sum()),
        f"{column} imputed with the couple median",
    )
    register(
        "DQ06",
        "schedule",
        "warning",
        int(by_plant.sum()),
        f"{column} imputed with the plant median",
    )
    register("DQ06", "schedule", "error", int(unresolved.sum()), f"{column} not imputable")
    return result


for measure in ("weight", "volume"):
    schedule = impute_with_median(schedule, measure)

schedule[schedule["weight_imputed"] | schedule["volume_imputed"]].head()

11:54:14 WARNING DQ06 [schedule] 108 row(s) - weight imputed with the couple median
11:54:14 WARNING DQ06 [schedule] 26 row(s) - weight imputed with the plant median
11:54:14 INFO    DQ06 [schedule] 0 row(s) - weight not imputable
11:54:14 WARNING DQ06 [schedule] 90 row(s) - volume imputed with the couple median
11:54:14 WARNING DQ06 [schedule] 26 row(s) - volume imputed with the plant median
11:54:14 INFO    DQ06 [schedule] 0 row(s) - volume not imputable


,date,supplier_id,material,plant,unloading_point,schedule_quantity,weight,volume,weight_imputed,weight_impute_level,volume_imputed,volume_impute_level
1755,2026-11-10,SUP_00112,MAT_000611,PLT_004,ULP_0017,18.0,1593.50,1.142400,True,couple,True,couple
2120,2026-11-09,SUP_00109,MAT_000522,PLT_002,ULP_0025,271.0,422.52,0.960000,True,couple,True,couple
2178,2026-12-08,SUP_00133,MAT_001031,PLT_002,ULP_0022,32.0,99.60,0.570000,True,plant,True,plant
2354,2026-11-09,SUP_00117,MAT_000613,PLT_002,ULP_0022,30.0,604.00,1.569586,True,couple,True,couple
2355,2026-11-10,SUP_00117,MAT_000613,PLT_002,ULP_0022,30.0,604.00,1.569586,True,couple,True,couple


### DQ07 - Ligne plus lourde qu'un camion
**Règle :** supprimer toute ligne dont le poids dépasse `truck_kg` (24 500 kg par défaut).
**Pourquoi :** le poids d'une ligne est celui d'un seul jour de besoin ; au-delà de la capacité d'un camion, la ligne ne tient pas dans
une livraison (RG04) et on choisit de ne pas la traiter. La règle passe après l'imputation (DQ06) : elle voit donc aussi les poids estimés.
Les lignes retirées sont comptées dans `rows_removed` de l'indicateur de précision.

In [9]:
overweight = schedule["weight"] > cfg.truck_kg
overweight_rows = schedule.loc[
    overweight, ["date", "supplier_id", "material", "plant", "schedule_quantity", "weight"]
]
register(
    "DQ07",
    "schedule",
    "warning",
    int(overweight.sum()),
    f"dropped (weight > {cfg.truck_kg:.0f} kg, heavier than one truck)",
)
schedule = schedule.loc[~overweight].copy()
overweight_rows

11:54:14 WARNING DQ07 [schedule] 1 row(s) - dropped (weight > 24500 kg, heavier than one truck)


,date,supplier_id,material,plant,schedule_quantity,weight
5015,2026-12-15,SUP_00074,MAT_001164,PLT_003,5670.0,26905.2


## 6. Distances

### DQ08 - Fournisseur en double
**Règle :** garder la première valeur et signaler le reste. **Pourquoi :** un fournisseur n'a qu'une distance.

### DQ09 - Distance manquante
**Règle :** une distance vide, absente de `distances.csv` ou `<= 0` est remplacée par la **médiane des distances des
fournisseurs présents dans le planning** (60,5 km dans `data_quality.ipynb`) et marquée `distance_imputed = True`.
**Pourquoi :** après DQ05, un seul fournisseur est concerné (SUP_00130, 4 lignes ; SUP_00047 n'apparaissait que le 01/01).
Garder ses besoins dans la consolidation coûte moins de précision que de les retirer.
**Alternative (option B) :** exclure ces fournisseurs du calcul de coût.

In [10]:
n_dup_suppliers = int(distances_raw["supplier_id"].duplicated().sum())
register("DQ08", "distances", "warning", n_dup_suppliers, "kept first value")
distances = distances_raw.drop_duplicates("supplier_id", keep="first").copy()
distances["distance"] = pd.to_numeric(distances["distance"], errors="coerce")
distances.loc[distances["distance"] <= 0, "distance"] = np.nan

planning_suppliers = pd.DataFrame({"supplier_id": sorted(schedule["supplier_id"].unique())})
supplier_distance = planning_suppliers.merge(distances, on="supplier_id", how="left")
fallback_distance = float(supplier_distance["distance"].median())

supplier_distance["distance_imputed"] = supplier_distance["distance"].isna()
supplier_distance["distance"] = supplier_distance["distance"].fillna(fallback_distance)

missing_suppliers = supplier_distance.loc[supplier_distance["distance_imputed"], "supplier_id"]
register(
    "DQ09",
    "distances",
    "warning",
    len(missing_suppliers),
    f"{len(missing_suppliers)} supplier(s) imputed with the median {fallback_distance} km",
)

schedule = schedule.merge(supplier_distance, on="supplier_id", how="left", validate="m:1")
logger.info("Fallback distance: %s km, suppliers: %s", fallback_distance, list(missing_suppliers))
supplier_distance[supplier_distance["distance_imputed"]]

11:54:14 INFO    DQ08 [distances] 0 row(s) - kept first value
11:54:14 WARNING DQ09 [distances] 1 row(s) - 1 supplier(s) imputed with the median 60.5 km
11:54:14 INFO    Fallback distance: 60.5 km, suppliers: ['SUP_00130']


,supplier_id,distance,distance_imputed
76,SUP_00130,60.5,True


## 7. Grille tarifaire

### DQ10
**Règle :** convertir en nombres et **arrêter** si un prix manque ou baisse quand la distance ou le poids augmente.
**Pourquoi :** la lecture du prix (RG11) prend la tranche supérieure ; un trou ou une inversion produirait en silence un mauvais coût.
`>4000` est un tarif forfaitaire par envoi.

In [11]:
rates = rates_raw.set_index("km_up_to").astype(float)

n_missing_prices = int(rates.isna().sum().sum())
n_dist_inversions = int((rates.diff().iloc[1:] < 0).sum().sum())
n_weight_inversions = int((rates.T.diff().iloc[1:] < 0).sum().sum())

register("DQ10", "rate_sheet", "error", n_missing_prices, "missing price")
register("DQ10", "rate_sheet", "error", n_dist_inversions, "price decreases with distance")
register("DQ10", "rate_sheet", "error", n_weight_inversions, "price decreases with weight")

if n_missing_prices or n_dist_inversions or n_weight_inversions:
    raise ValueError("Rate sheet is inconsistent, see the anomaly register")
rates.iloc[:3, :5]

11:54:14 INFO    DQ10 [rate_sheet] 0 row(s) - missing price
11:54:14 INFO    DQ10 [rate_sheet] 0 row(s) - price decreases with distance
11:54:14 INFO    DQ10 [rate_sheet] 0 row(s) - price decreases with weight


,50,100,200,300,400
km_up_to,,,,,
30,4.75,6.66,8.56,10.73,13.06
50,5.78,7.94,10.74,13.07,15.45
75,6.92,9.54,13.14,15.99,18.43


## 8. Indicateur de précision des données (accuracy)

Toutes les modifications faites ci-dessus sont comptées dans la précision de l'optimisation : plus une part
des résultats repose sur des valeurs imputées, moins ils sont fiables.

| Indicateur | Définition |
| :--- | :--- |
| `rows_removed` | Lignes brutes retirées du périmètre (DQ01 à DQ05 et DQ07) |
| `rows_modified` | Lignes avec au moins une valeur imputée : poids, volume (DQ06) ou distance du fournisseur (DQ09) |
| `rows_accuracy` | 1 - `rows_modified` / `rows` : part des lignes entièrement d'origine |
| `weight_accuracy` | Part du poids total qui vient de valeurs d'origine (non imputées) |

Les mêmes indicateurs sont calculés globalement, par usine et par couple fournisseur × usine (unité de l'optimisation).

In [12]:
def accuracy(df: pd.DataFrame) -> pd.Series:
    """Share of original (non-imputed) rows and weight in `df`."""
    modified = df["weight_imputed"] | df["volume_imputed"] | df["distance_imputed"]
    original_weight = df["weight"].where(~df["weight_imputed"], 0.0)
    return pd.Series(
        {
            "rows": len(df),
            "rows_modified": int(modified.sum()),
            "rows_accuracy": 1 - float(modified.mean()),
            "weight_accuracy": float(original_weight.sum() / df["weight"].sum()),
        }
    )


accuracy_global = accuracy(schedule)
accuracy_global["rows_removed"] = len(schedule_raw) - len(schedule)
accuracy_by_plant = pd.DataFrame({k: accuracy(g) for k, g in schedule.groupby("plant")}).T
accuracy_by_couple = pd.DataFrame(
    {k: accuracy(g) for k, g in schedule.groupby(["supplier_id", "plant"])}
).T
accuracy_by_couple.index.names = ["supplier_id", "plant"]

logger.info(
    "Accuracy: rows %.1f%% | weight %.1f%% | %d row(s) modified, %d removed",
    100 * accuracy_global["rows_accuracy"],
    100 * accuracy_global["weight_accuracy"],
    accuracy_global["rows_modified"],
    accuracy_global["rows_removed"],
)
accuracy_by_plant.round(4)

11:54:14 INFO    Accuracy: rows 96.9% | weight 96.7% | 138 row(s) modified, 1339 removed


,rows,rows_modified,rows_accuracy,weight_accuracy
PLT_001,1183.0,25.0,0.9789,0.9563
PLT_002,1419.0,55.0,0.9612,0.9531
PLT_003,448.0,28.0,0.9375,0.9508
PLT_004,1339.0,30.0,0.9776,0.9851


## 9. Résultat et export
La consolidation lit `schedule_clean.csv`, `supplier_distance.csv`, `anomaly_register.csv` et les trois tableaux de précision (`accuracy_*.csv`).

In [13]:
schedule_clean = schedule
anomaly_register = pd.DataFrame(ANOMALIES)

assert schedule_clean[["weight", "volume", "distance"]].notna().all().all()
assert (schedule_clean["schedule_quantity"] > 0).all()
assert not schedule_clean.duplicated(list(cfg.key_columns)).any()

logger.info("Raw: %d rows -> clean: %d rows", len(schedule_raw), len(schedule_clean))

cfg.output_dir.mkdir(parents=True, exist_ok=True)
schedule_clean.to_csv(cfg.output_dir / "schedule_clean.csv", index=False)
supplier_distance.to_csv(cfg.output_dir / "supplier_distance.csv", index=False)
anomaly_register.to_csv(cfg.output_dir / "anomaly_register.csv", index=False)
accuracy_global.to_frame("value").to_csv(cfg.output_dir / "accuracy_global.csv")
accuracy_by_plant.to_csv(cfg.output_dir / "accuracy_by_plant.csv")
accuracy_by_couple.to_csv(cfg.output_dir / "accuracy_by_couple.csv")
logger.info("Files written to %s", cfg.output_dir.resolve())

anomaly_register

11:54:14 INFO    Raw: 5728 rows -> clean: 4389 rows
11:54:14 INFO    Files written to C:\Users\houeg\projects\delivery_schedule_optimization\data


,rule,source,severity,rows,action
0,DQ01,schedule,none,0,trimmed / upper-cased
1,DQ01,schedule,none,0,dropped (invalid identifier)
2,DQ02,schedule,none,0,dropped (exact duplicate)
3,DQ02,schedule,none,0,"flagged only (same key, different values)"
4,DQ03,schedule,info,959,"dropped (quantity = 0, no need)"
5,DQ03,schedule,none,0,dropped (negative quantity)
6,DQ03,schedule,none,0,dropped (missing quantity)
7,DQ04,schedule,none,0,flagged and kept (outside horizon)
8,DQ04,schedule,none,0,"flagged and kept (no delivery on weekends, han..."
9,DQ04,schedule,none,0,flagged and kept (no delivery on German public...
